# 16 Sensitivitas Candidate Budget


In [ ]:
# Jalur REVISED dan pembaca hasil praproses.
from pathlib import Path
import os, sys
from IPython.display import display
try:
    from google.colab import drive
except ImportError:
    pass
else:
    if not Path('/content/drive/MyDrive').is_dir():drive.mount('/content/drive',force_remount=False)
_explicit=os.environ.get('CARE_KM_ROOT') or os.environ.get('CARE_KM_PAPER_ROOT')
_options=([Path(_explicit)] if _explicit else []) + [Path('/content/drive/MyDrive/REVISED'),
    Path(__file__).resolve().parent.parent if '__file__' in globals() else Path.cwd(),Path.cwd(),Path.cwd().parent]
_ROOT_REVISED=next((p.resolve() for p in _options if all((p/d).is_dir() for d in ('DATA','LIB','PYTHON'))),None)
if _ROOT_REVISED is None:raise FileNotFoundError('Letakkan DATA, LIB, PYTHON di MyDrive/REVISED.')
sys.path.insert(0,str(_ROOT_REVISED/'LIB'))
# Jangan memakai modul versi lama yang masih tersimpan di memori Colab.
import importlib
_loaded=sys.modules.get('pipeline_paths')
if _loaded is None or (getattr(_loaded,'PIPELINE_LAYOUT_VERSION',None)!='2026.11.06-experiments-3'
    or Path(getattr(_loaded,'__file__','')).resolve()!=(_ROOT_REVISED/'LIB/pipeline_paths.py').resolve()):
    for _mod in ('pipeline_paths','pipeline_data','pipeline_runner','pipeline_cache','experiment_lib','care_km','generalization_data'):
        sys.modules.pop(_mod,None)
    importlib.invalidate_caches()
from pipeline_paths import configure,stage_dir,lms_data,pendikom_data,artifact_path,record_resample
configure(_ROOT_REVISED)


In [ ]:
from pipeline_data import preflight
preflight(_ROOT_REVISED,benchmarks=False)


In [ ]:
from pathlib import Path
import json
import os
import sys
import time
import hashlib

import numpy as np
import pandas as pd
from scipy.stats import wilcoxon

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except Exception:
    pass

# Locate CARE-KM-PAPER root.
_candidates = [
    Path(os.environ.get('CARE_KM_PAPER_ROOT', '')) if os.environ.get('CARE_KM_PAPER_ROOT') else None,
    _ROOT_REVISED,
    _ROOT_REVISED,
    Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent,
]
PAPER_ROOT = next(
    (p.resolve() for p in _candidates if p is not None and (p/'LIB').is_dir() and (p/'DATA').is_dir()),
    None,
)
if PAPER_ROOT is None:
    raise FileNotFoundError('Root REVISED tidak ditemukan.')

LIB_DIR = PAPER_ROOT / 'LIB'
DATA_DIR = PAPER_ROOT / 'DATA'
OUT_DIR = stage_dir('16_Sensitivitas_Candidate_Budget')
OUT_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(LIB_DIR))

import experiment_lib as el
import care_km as ck

K = el.K
MAX_ITER = el.MAX_ITER
SI_SAMPLE_SIZE = el.SI_SAMPLE_SIZE

print('PAPER_ROOT:', PAPER_ROOT)
print('CARE implementation:', getattr(ck, 'CARE_IMPLEMENTATION_VERSION', 'unknown'))
print('K:', K, '| MAX_ITER:', MAX_ITER, '| SI sample size:', SI_SAMPLE_SIZE)
print('Frozen CARE configuration:', el.FROZEN_CARE_CONFIG)

# Protocol: exactly the same matched-resample design as notebook 05.
Q_VALUES = [2, 4, 8, 16, 32, 64]
REPS = 20
FRACTION = 0.80
BASE_SEED = 5161600

CANDIDATE_RULE = str(el.FROZEN_CARE_CONFIG['candidate_rule'])
MIN_GAIN = float(el.FROZEN_CARE_CONFIG['min_gain'])

# q=16 is the pre-specified manuscript operating point.
PRIMARY_Q = 16
assert PRIMARY_Q in Q_VALUES
EXPECTED_BUDGETS = {2: 1, 4: 2, 8: 3, 16: 4, 32: 5, 64: 6}
for q, expected in EXPECTED_BUDGETS.items():
    actual = int(ck.derive_refinement_budget(q))
    assert actual == expected, f'q={q}: expected E_max={expected}, got {actual}'

RAW_PATH = artifact_path('18_q_sensitivity_raw.csv','16_Sensitivitas_Candidate_Budget')
SUMMARY_PATH = artifact_path('18_q_sensitivity_summary.csv','16_Sensitivitas_Candidate_Budget')
PAIRED_PATH = artifact_path('18_q_sensitivity_paired_vs_q16.csv','16_Sensitivitas_Candidate_Budget')
PROTOCOL_PATH = artifact_path('18_q_sensitivity_protocol.json','16_Sensitivitas_Candidate_Budget')
INTERPRETATION_PATH = artifact_path('18_q_sensitivity_interpretation.txt','16_Sensitivitas_Candidate_Budget')

protocol = {
    'purpose': 'diagnostic sensitivity analysis; not hyperparameter tuning or model selection',
    'q_values': Q_VALUES,
    'primary_q': PRIMARY_Q,
    'refinement_budget_rule': 'ceil(log2(candidate_budget))',
    'derived_budgets': {str(q): int(ck.derive_refinement_budget(q)) for q in Q_VALUES},
    'repetitions': REPS,
    'fraction': FRACTION,
    'base_seed': BASE_SEED,
    'matched_resamples': True,
    'K': int(K),
    'candidate_rule': CANDIDATE_RULE,
    'min_gain': MIN_GAIN,
    'max_kmodes_iterations': int(MAX_ITER),
    'si_sample_size': int(SI_SAMPLE_SIZE),
    'care_implementation_version': getattr(ck, 'CARE_IMPLEMENTATION_VERSION', 'unknown'),
    'frozen_q16_metadata': el.frozen_care_metadata(),
}
PROTOCOL_PATH.write_text(json.dumps(protocol, indent=2), encoding='utf-8')
print(json.dumps(protocol, indent=2))


# Load and audit the exact frozen categorical LMS dataset.
df, X_all, FEATURES = el.load_lms_dataset(PAPER_ROOT)

dataset_path = lms_data('categorical')
dataset_sha256 = el.sha256_file(dataset_path)
care_source_path = LIB_DIR / 'care_km.py'
care_sha256 = el.sha256_file(care_source_path)

print('Dataset:', dataset_path)
print('Rows:', len(df), '| Features:', len(FEATURES))
print('Dataset SHA-256:', dataset_sha256)
print('care_km.py SHA-256:', care_sha256)

audit = el.dataset_audit(df, FEATURES)
display(audit)

if len(df) != 6658:
    raise RuntimeError(f'Expected 6658 learners, got {len(df)}')
if len(FEATURES) != 13:
    raise RuntimeError(f'Expected 13 features, got {len(FEATURES)}')

# Update protocol with source hashes.
protocol.update({
    'dataset_path': str(dataset_path),
    'dataset_sha256': dataset_sha256,
    'care_source_path': str(care_source_path),
    'care_source_sha256': care_sha256,
})
PROTOCOL_PATH.write_text(json.dumps(protocol, indent=2), encoding='utf-8')

# Resume-safe execution.
if RAW_PATH.exists():
    raw = pd.read_csv(RAW_PATH)
else:
    raw = pd.DataFrame()

expected_pairs = {(rep, q) for rep in range(REPS) for q in Q_VALUES}
if len(raw):
    completed_pairs = set(
        zip(
            pd.to_numeric(raw['rep'], errors='coerce').astype('Int64'),
            pd.to_numeric(raw['q'], errors='coerce').astype('Int64'),
        )
    )
else:
    completed_pairs = set()

for rep in range(REPS):
    rng = np.random.default_rng(BASE_SEED + rep)
    idx = np.sort(
        rng.choice(
            len(X_all),
            size=int(len(X_all) * FRACTION),
            replace=False,
        )
    )
    X = X_all[idx]
    sample_hash = el.sample_index_hash(idx)
    record_resample('16_Sensitivitas_Candidate_Budget','LMS',rep,idx,sample_hash)

    print(f'\nrep={rep+1:02d}/{REPS} | n={len(X)} | sample_hash={sample_hash}')

    for q in Q_VALUES:
        if (rep, q) in completed_pairs:
            print(f'  q={q}: SKIP checkpoint')
            continue

        care_config = {
            'candidate_rule': CANDIDATE_RULE,
            'candidate_budget': int(q),
            'min_gain': MIN_GAIN,
        }
        derived_budget = int(ck.derive_refinement_budget(q))

        t0 = time.perf_counter()
        init_modes, trace = ck.care_init(
            X,
            K,
            return_trace=True,
            **care_config,
        )
        t1 = time.perf_counter()

        fit = ck.kmodes_fit(X, init_modes, max_iter=MAX_ITER)
        t2 = time.perf_counter()

        metrics = ck.evaluate_clustering(
            X,
            None,
            fit,
            init_modes,
            si_sample_size=SI_SAMPLE_SIZE,
            si_seed=BASE_SEED + rep,
        )
        t3 = time.perf_counter()

        accepted = int(len(trace))
        stopping_reason = (
            'NO_POSITIVE_GAIN'
            if accepted < derived_budget
            else 'DERIVED_REFINEMENT_BUDGET_REACHED'
        )

        row = {
            'rep': rep,
            'rep_display': rep + 1,
            'q': int(q),
            'derived_refinement_budget': derived_budget,
            'accepted_corrections': accepted,
            'stopping_reason': stopping_reason,
            'sample_n': int(len(X)),
            'sample_hash': sample_hash,
            **metrics,
            'care_init_sec': float(t1 - t0),
            'kmodes_sec': float(t2 - t1),
            'evaluation_sec': float(t3 - t2),
            'algorithm_sec': float(t2 - t0),
            'total_sec': float(t3 - t0),
        }

        raw = pd.concat([raw, pd.DataFrame([row])], ignore_index=True)
        raw = raw.sort_values(['rep','q']).reset_index(drop=True)
        el.save_csv(raw, RAW_PATH)
        completed_pairs.add((rep, q))

        print(
            f"  q={q:2d} | Emax={derived_budget} | accepted={accepted} | "
            f"init={metrics['initial_cost']:.1f} | final={metrics['final_cost']:.1f} | "
            f"DBI={metrics['dbi']:.4f} | SI={metrics['si']:.4f} | "
            f"iter={metrics['iterations']} | alg_sec={row['algorithm_sec']:.2f}"
        )

# Integrity checks.
actual_pairs = set(zip(raw['rep'].astype(int), raw['q'].astype(int)))
missing = sorted(expected_pairs - actual_pairs)
if missing:
    raise RuntimeError(f'Checkpoint belum lengkap. Missing rep/q pairs: {missing[:10]}')

if raw.duplicated(['rep','q']).any():
    raise RuntimeError('Duplicate rep/q rows found.')

# Every q within a repetition must use the same matched sample.
if not (raw.groupby('rep')['sample_hash'].nunique() == 1).all():
    raise RuntimeError('Matched-resample integrity FAILED: sample hashes differ across q.')

print('\nMatched-resample integrity: PASS')
print('Complete rows:', len(raw), 'expected:', REPS * len(Q_VALUES))
display(raw.head(9))

summary = (
    raw.groupby(['q','derived_refinement_budget'], as_index=False)
    .agg(
        n=('rep','size'),
        accepted_corrections_mean=('accepted_corrections','mean'),
        accepted_corrections_sd=('accepted_corrections','std'),
        initial_cost_mean=('initial_cost','mean'),
        initial_cost_sd=('initial_cost','std'),
        final_cost_mean=('final_cost','mean'),
        final_cost_sd=('final_cost','std'),
        dbi_mean=('dbi','mean'),
        dbi_sd=('dbi','std'),
        si_mean=('si','mean'),
        si_sd=('si','std'),
        iterations_mean=('iterations','mean'),
        iterations_sd=('iterations','std'),
        care_init_sec_mean=('care_init_sec','mean'),
        algorithm_sec_mean=('algorithm_sec','mean'),
        total_sec_mean=('total_sec','mean'),
    )
    .sort_values('q')
    .reset_index(drop=True)
)

# Add early-stopping frequency.
early = (
    raw.assign(early_stop=raw['stopping_reason'].eq('NO_POSITIVE_GAIN'))
    .groupby('q')['early_stop']
    .agg(['sum','mean'])
    .reset_index()
    .rename(columns={'sum':'early_stop_n','mean':'early_stop_rate'})
)
summary = summary.merge(early, on='q', how='left')

el.save_csv(summary, SUMMARY_PATH)
display(summary)
print('Saved:', SUMMARY_PATH)

METRIC_DIRECTIONS = {
    'initial_cost': 'lower',
    'final_cost': 'lower',
    'dbi': 'lower',
    'si': 'higher',
    'iterations': 'lower',
    'algorithm_sec': 'lower',
}

rows = []
for metric, direction in METRIC_DIRECTIONS.items():
    base = raw.loc[raw['q'].eq(PRIMARY_Q), ['rep', metric]].set_index('rep')[metric].sort_index()
    metric_rows = []

    for q_cmp in [q for q in Q_VALUES if q != PRIMARY_Q]:
        comp = raw.loc[raw['q'].eq(q_cmp), ['rep', metric]].set_index('rep')[metric].sort_index()
        common = base.index.intersection(comp.index)
        a = base.loc[common].to_numpy(dtype=float)   # q=16
        b = comp.loc[common].to_numpy(dtype=float)  # comparator
        diff = a - b

        if direction == 'lower':
            wins = int(np.sum(a < b))
            losses = int(np.sum(a > b))
        else:
            wins = int(np.sum(a > b))
            losses = int(np.sum(a < b))
        ties = int(np.sum(np.isclose(a, b, rtol=0.0, atol=1e-12)))

        # Wilcoxon can fail if every paired difference is exactly zero.
        if np.all(np.isclose(diff, 0.0, atol=1e-12, rtol=0.0)):
            stat, p = 0.0, 1.0
        else:
            stat, p = wilcoxon(a, b, alternative='two-sided', zero_method='wilcox')

        metric_rows.append({
            'metric': metric,
            'direction': direction,
            'primary_q': PRIMARY_Q,
            'comparator_q': q_cmp,
            'n_pairs': len(common),
            'q16_mean': float(np.mean(a)),
            'comparator_mean': float(np.mean(b)),
            'mean_difference_q16_minus_comparator': float(np.mean(diff)),
            'median_difference_q16_minus_comparator': float(np.median(diff)),
            'q16_wins': wins,
            'ties': ties,
            'q16_losses': losses,
            'wilcoxon_statistic': float(stat),
            'wilcoxon_p_two_sided': float(p),
        })

    adj = el.holm_adjust([r['wilcoxon_p_two_sided'] for r in metric_rows])
    for r, p_adj in zip(metric_rows, adj):
        r['holm_p_within_metric'] = float(p_adj)
        rows.append(r)

paired = pd.DataFrame(rows)
el.save_csv(paired, PAIRED_PATH)
display(paired)
print('Saved:', PAIRED_PATH)

# Relative spread across the full q grid for each main metric.
main_metrics = ['initial_cost','final_cost','dbi','si','iterations','algorithm_sec']
stability_rows = []
summary_by_q = summary.set_index('q')

for metric in main_metrics:
    col = f'{metric}_mean'
    if col not in summary_by_q.columns:
        continue

    means = summary_by_q[col]
    vals = means.loc[Q_VALUES].to_numpy(dtype=float)
    denom = abs(float(means.loc[PRIMARY_Q]))
    relative_range_pct = (
        float(vals.max() - vals.min()) / denom * 100.0
        if denom > 0 else np.nan
    )

    row = {
        'metric': metric,
        'q16_mean': float(means.loc[PRIMARY_Q]),
        'min_mean_across_q': float(vals.min()),
        'max_mean_across_q': float(vals.max()),
        'range_across_q': float(vals.max() - vals.min()),
        'range_relative_to_q16_pct': relative_range_pct,
    }
    for q in Q_VALUES:
        row[f'q{q}_mean'] = float(means.loc[q])
    stability_rows.append(row)

stability = pd.DataFrame(stability_rows)
display(stability)


# Generate a concise interpretation file WITHOUT declaring an optimum.
# This text is a diagnostic summary; authors should inspect the tables before using it in the paper.
s = summary.set_index('q')
lines = []
lines.append('CARE-KM candidate-budget sensitivity: q={2,4,8,16,32,64}')
lines.append('Purpose: diagnostic sensitivity analysis; not hyperparameter tuning or model selection.')
lines.append('Matched design: 20 identical 80% LMS resamples for every q.')
lines.append('Derived budgets: q=2->1, q=4->2, q=8->3, q=16->4, q=32->5, q=64->6.')
lines.append('Reference operating point: q=16 (pre-specified manuscript setting).')
lines.append('')

for q in Q_VALUES:
    r = s.loc[q]
    lines.append(
        f"q={q}: init={r['initial_cost_mean']:.3f}, final={r['final_cost_mean']:.3f}, "
        f"DBI={r['dbi_mean']:.6f}, SI={r['si_mean']:.6f}, "
        f"iter={r['iterations_mean']:.3f}, accepted={r['accepted_corrections_mean']:.3f}, "
        f"algorithm_sec={r['algorithm_sec_mean']:.3f}"
    )

lines.append('')
lines.append(
    'Interpretation rule: q=16 must be described as the pre-specified intermediate search budget, not as an optimum. '
    'Use the wider sensitivity grid to determine whether smaller budgets materially degrade post-convergence behavior '
    'and whether larger budgets provide diminishing post-convergence gains relative to their additional computational cost.'
)
lines.append(
    'Do not choose a new q from this table solely because it has the best DBI, SI, or cost. '
    'If q=16 is retained, justify it as a bounded operating point supported by stability/trade-off evidence.'
)

INTERPRETATION_PATH.write_text('\n'.join(lines), encoding='utf-8')
print(INTERPRETATION_PATH.read_text(encoding='utf-8'))
